# Challenge 01 - Meet Agent Brut Force

**Draft - not tested.** Goal: create a Foundry prompt agent using Elastic's hosted MCP server, then explain its discovery trace. The agent definition and tool configuration are visible below; the shared helper only handles approvals, capture, and measurement.

Predict which source the first question needs before running it. Do not instruct the agent to inspect every index.

In [ ]:
from pathlib import Path
import sys
resource_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(resource_root))
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from workshop import configure, required, workshop_id, tool_names, discover_tools, mcp_url, foundry_clients, base_prompt, remember_agent, run_agent, read_json, save_json, summarize, trace
configure()
project, client = foundry_clients()
advertised = await discover_tools('baseline')
display([tool['name'] for tool in advertised])

## Configure

Choose the list-indices, mapping, and ES|QL wire names from discovery. Fill `selected_tools` below. The corresponding `.env` entries must match so trace classification is consistent. Why should the support agent not receive workflow execution or write tools?

Read the provided system prompt. It scopes the sources, limits results, and asks for citations; it does not force an inefficient route. The API-safe agent name uses hyphens for the workshop's Agent Brut Force.

In [ ]:
selected_tools = []  # Student decision: exact discovery and read-only query wire names.
if set(selected_tools) != set(tool_names('baseline')):
    raise ValueError('Choose the three baseline tools and match their .env names.')
if not set(selected_tools).issubset({tool['name'] for tool in advertised}):
    raise ValueError('A selected tool was not advertised by Elastic MCP.')
instructions = base_prompt()
print(instructions)
tool = MCPTool(
    server_label='elastic', server_url=mcp_url(),
    project_connection_id=required('ELASTIC_BASELINE_CONNECTION'),
    allowed_tools=selected_tools, require_approval='always',
)
agent = project.agents.create_version(
    agent_name=f'Agent-Brut-Force-{workshop_id()}',
    definition=PromptAgentDefinition(
        model=required('AZURE_AI_MODEL_DEPLOYMENT_NAME'),
        instructions=instructions, tools=[tool],
    ),
)
remember_agent(agent, 'baseline')
print('Created:', agent.name, 'version', agent.version)

## Observe

Run the same fixed question set that Optimus will receive. Every helper invocation creates a fresh conversation. Review the server, tool, and arguments for each approval; approve only expected read-only calls. The calls execute remotely in Foundry, not as local Python Elasticsearch queries.

Elapsed time includes your approval pauses and is not a pure service-latency comparison. Response token usage comes from the service and includes each continuation exactly once; missing values remain unavailable. Inspect raw cached-token details when interpreting costs.

In [ ]:
questions = read_json('data/questions.json')
baseline_runs = []
for question in questions:
    print('\nQuestion:', question['query'])
    result = run_agent(client, agent, question['query'], selected_tools)
    result['question_id'] = question['id']
    baseline_runs.append(result)
    save_json('baseline-runs.json', baseline_runs)
    print(result['response'])
    display(summarize(result))

## Explain

Inspect a successful trace. Which calls discover structure, and which retrieve facts? Verify citations against the returned records. Does an invoice question have enough evidence for a numeric answer?

Fill your own quality notes before interpreting efficiency. A plausible answer without supporting retrieval is not a successful run.

In [ ]:
display(trace(baseline_runs[0]))
quality_notes = {
    question['id']: {'correct': None, 'citations_supported': None, 'explanation': ''}
    for question in questions
}
display(quality_notes)
# Edit the quality notes using actual retrieval evidence, then save them.
save_json('baseline-quality.json', quality_notes)